#Ingest circuits.csv file
1.  Read the file using spark dataframe reader API
2. Add Metadata Columns
    * Source file
    * Ingestion Timestamp
3. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
catlog_name

In [0]:

landing_folder_path

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catlog_name}.{bronze_schema}.circuits"

In [0]:
source_file

#Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuit_schema = StructType([
    StructField('circuitID', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType()),
    

])

In [0]:
circuit_df = (
    spark.read
              .format('csv')
              .option('header', 'true')
              .option('inferSchema', 'true')
              .schema(circuit_schema)
              .option('mode', 'FAILFAST')
              .load(source_file)
)


In [0]:
display(circuit_df)

#Step 2 - Add Metadata Columns
* Source File
* Ingestion TimeStamp

In [0]:
circuit_final_df = add_ingestion_metadata(circuit_df)

In [0]:
display(circuit_final_df)

#Step 3 - Write to bronze delta table

In [0]:
(
    circuit_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
%sql
SELECT * from formula1.bronze.circuits

In [0]:
display(spark.table('formula1.bronze.circuits'))